# **Decoder Casual Language Model**

In [1]:
import torchtext
import torch
from torchtext.data.utils import get_tokenizer
from torchtext.vocab import build_vocab_from_iterator
from torchtext.datasets import IMDB

C:\Users\Vish\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### **Dataset**

In [2]:
train_iter,test_iter = IMDB()
next(iter(train_iter))

(1,
 'I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far betwee

In [3]:
# define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device.type

'cpu'

## Preprocessing Data

This section focuses on preparing text data for NLP tasks, mainly through tokenization and vocabulary creation.

### Special Tokens and Indices
Defines special tokens such as `<unk>`, `<pad>`, and an empty string (used as an EOS token), and assigns them indices (`0`, `1`, and `2`). These tokens serve specific purposes in text processing:

- `UNK_IDX`: Represents unknown or out-of-vocabulary words  
- `PAD_IDX`: Used to pad shorter sequences in a batch for uniform length  
- `EOS_IDX`: Indicates the end of a sequence  

### `yield_tokens` Function
A generator function that iterates through the dataset (`data_iter`), applies a tokenizer to each sample, and yields tokenized outputs one at a time.

### Building the Vocabulary
Constructs a vocabulary from the tokenized data using `build_vocab_from_iterator`. Special tokens are added at the beginning, and all tokens with a minimum frequency of 1 are included.

### Handling Unknown Tokens
Sets a default index (`UNK_IDX`) for tokens not found in the vocabulary, ensuring the model can handle unseen words gracefully.

### `text_to_index` Function
Converts raw text into a sequence of numerical indices based on the vocabulary, making it suitable as input for machine learning models.

### `index_to_en` Function
Transforms a sequence of indices back into readable text, which is useful for interpreting model outputs.

### Functionality Check
Validates the preprocessing pipeline by converting a sample tensor `[0, 1, 2]` back into the corresponding special tokens, ensuring that the mappings are working correctly.

In [4]:
UNK_IDX, PAD_IDX, EOS_IDX = 0, 1, 2
special_tokens = ["<unk>","<pad>", "<eos>" ]

In [5]:
tokenizer = get_tokenizer("basic_english")

def yield_tokens(dataset):
    for id, text in dataset:
        yield tokenizer(text)
        
vocab = build_vocab_from_iterator(yield_tokens(train_iter),specials=special_tokens,
                                  special_first=True)
vocab.set_default_index(UNK_IDX)

In [6]:
vocab["store"]

1024

#### **Build Pipelines**

Text to Indices

In [7]:
def text_pipeline (sample):
    return [vocab[text] for text in tokenizer(sample) ]

text_pipeline("I rented I AM CURIOUS-YELLOW from my video store because of all")

[13, 1153, 13, 230, 24141, 50, 72, 351, 1024, 87, 9, 40]

Indices to Text

In [8]:
vocab.get_itos()[1153]

'rented'

In [9]:
def index_to_en (indices):
    tokens_list = [vocab.get_itos()[index] for index in indices]
    tokens_list = " ".join(tokens_list)
    return tokens_list
    
index_to_en([13, 1153, 13, 230, 24141, 50, 72, 351, 1024, 87, 9, 40])

'i rented i am curious-yellow from my video store because of all'

### Collate Function

For the decoder model, a collate function is designed to process batches of text data. This function takes a block of text as input and returns a transformed version suitable for training.

The transformation is handled by the `get_sample(block_size, text)` function. This function randomly extracts a segment of text (`src_sequence`) along with its shifted counterpart (`tgt_sequence`), which represents the next tokens in the sequence.

It ensures that the sampled text fits within the specified block size. If the original text is shorter than the block size, adjustments are made accordingly. The function ultimately returns both the source and target sequences, which are used as inputs for training the language model.

In [12]:
def get_sample(block_size, text):
    
    text_len = len(text)
    stop_point = 0
    start_point = 0
    seq_differ = text_len-block_size 
    
    if (seq_differ >= 1):
        start_point = torch.randint(low=0, high = seq_differ, size=(1,)).item() 
        stop_point = start_point + block_size
        
        src_sequence = text[start_point:stop_point]
        tgt_sequence = text[start_point+ 1: stop_point+1]
    
    elif (seq_differ <= 0):
        start_point = 0
        stop_point = text_len
        
        src_sequence = text[start_point:stop_point]
        tgt_sequence = text[start_point+1: stop_point]
        
        tgt_sequence.append('<|endoftext|>')
    
    return src_sequence, tgt_sequence
    

In [34]:
batch_size = 1

batch_of_tokens = []
for i in range(batch_size):
    _,text = next(iter(train_iter))
    batch_of_tokens.append(tokenizer(text))
    

In [35]:
text = batch_of_tokens[0][0:100]
text[0:50]

['i',
 'rented',
 'i',
 'am',
 'curious-yellow',
 'from',
 'my',
 'video',
 'store',
 'because',
 'of',
 'all',
 'the',
 'controversy',
 'that',
 'surrounded',
 'it',
 'when',
 'it',
 'was',
 'first',
 'released',
 'in',
 '1967',
 '.',
 'i',
 'also',
 'heard',
 'that',
 'at',
 'first',
 'it',
 'was',
 'seized',
 'by',
 'u',
 '.',
 's',
 '.',
 'customs',
 'if',
 'it',
 'ever',
 'tried',
 'to',
 'enter',
 'this',
 'country',
 ',',
 'therefore']

In [36]:
block_size = 10
src_seq, tgt_seq = get_sample(block_size, text)
src_seq,tgt_seq

(['lena',
  'who',
  'wants',
  'to',
  'learn',
  'everything',
  'she',
  'can',
  'about',
  'life'],
 ['who',
  'wants',
  'to',
  'learn',
  'everything',
  'she',
  'can',
  'about',
  'life',
  '.'])